# Fireworks serverless RL on Countdown (step by step)

This notebook mirrors the [Fireworks cookbook `serverless_rl`](https://github.com/fw-ai/cookbook/tree/main/training/examples/serverless_rl) example, but **spells out every stage in the notebook** so you can read, run, and explain what happens under the hood.

**What “serverless” means here:** you do not provision a dedicated trainer job or inference deployment. One `FiretitanServiceClient` pointed at `/training/v1/serverless` gives you:

1. a **LoRA training client** (`create_lora_training_client`), and
2. a **sampling client** bound to a weight snapshot you just saved (`save_weights_for_sampler` → `create_sampling_client`).

Each optimizer step in the cookbook follows the same pattern:

```
save LoRA snapshot → sample group_size completions per prompt → score → GRPO advantages → importance_sampling forward/backward → Adam step
```

We use the bundled 32-row `countdown_train.jsonl`, keep **Router Replay** requested (MoE models replay expert routes from sampling during training), and guard all paid API calls behind `RUN_TRAINING = False`.

> **Cost:** cells through setup, rewards, and data prep are free. Setting `RUN_TRAINING = True` opens a real serverless session and charges for sampling + training.


## 0. Install the cookbook `training` package (SDK + helpers)

We still install `fw-ai/cookbook/training` because the notebook relies on the Fireworks SDK, chat renderers, tokenizer loaders, and Router Replay utilities—the same dependencies the upstream script uses. The **logic below is written out explicitly** rather than calling `ServerlessCountdownRL(...).run()`.


In [ ]:
%%bash
set -euxo pipefail
REPO_DIR=/content/fw-cookbook
rm -rf "${REPO_DIR}"
git clone --filter=blob:none --no-checkout https://github.com/fw-ai/cookbook.git "${REPO_DIR}"
git -C "${REPO_DIR}" sparse-checkout init --no-cone
git -C "${REPO_DIR}" sparse-checkout set /training/
git -C "${REPO_DIR}" checkout main
echo "Cookbook commit: $(git -C "${REPO_DIR}" rev-parse HEAD)"


In [ ]:
%cd /content/fw-cookbook
%pip install -q --pre -e training


## 1. Secrets and environment

The upstream example reads `FIREWORKS_API_KEY` from the environment (or `training/.env`). Kimi K3’s tokenizer also needs `HF_TRUST_REMOTE_CODE=1`.


In [ ]:
import os

os.environ["HF_TRUST_REMOTE_CODE"] = "1"

try:
    from google.colab import userdata
    api_key = userdata.get("FIREWORKS_API_KEY")
except Exception:
    api_key = os.environ.get("FIREWORKS_API_KEY", "")

if api_key:
    os.environ["FIREWORKS_API_KEY"] = api_key
    print("FIREWORKS_API_KEY is set.")
else:
    print("No API key yet — fine until you enable training.")


## 2. Hyperparameters (same defaults as `countdown_rl.py`)

These names match the cookbook `Config` dataclass. With the 32-row sample file, `eval_prompt_groups=16` holds out half the rows for fixed evaluation; the other 16 rows cycle for training.


In [ ]:
from pathlib import Path

REPO_ROOT = Path("/content/fw-cookbook")
DATASET = REPO_ROOT / "training/examples/serverless_rl/data/countdown_train.jsonl"
RUN_DIR = Path("/content/serverless_rl_run")

# Model
BASE_MODEL = "accounts/fireworks/models/kimi-k3"
TOKENIZER_MODEL = "moonshotai/Kimi-K3"
RENDERER_NAME = ""  # auto-resolve from tokenizer
LORA_RANK = 32
LORA_ALPHA = 64
MAX_SEQ_LEN = 32768

# Data
SHUFFLE = True
SEED = 0
EVAL_PROMPT_GROUPS = 16
EVAL_SEED = 1

# RL loop
STEPS = 20
PROMPT_GROUPS_PER_STEP = 16
GROUP_SIZE = 8
PROMPT_CONCURRENCY = 8
MAX_SAMPLE_TOKENS = 4096
TEMPERATURE = 1.0
LEARNING_RATE = 1e-4
ROUTER_REPLAY = True  # cookbook default; dense models skip automatically

# Fixed eval
EVAL_INTERVAL = 5
EVAL_GROUP_SIZE = 8
EVAL_AT_START = True
EVAL_AT_END = True

# Bookkeeping
API_BASE_URL = os.environ.get("FIREWORKS_BASE_URL", "https://api.fireworks.ai")
CHECKPOINT_NAME = "cd-sample"
FINAL_CHECKPOINT_NAME = "countdown-final"
SAMPLING_TIMEOUT_S = 1800.0

assert DATASET.is_file(), DATASET
assert ROUTER_REPLAY is True
print(f"Dataset: {DATASET}")
print(f"Run dir: {RUN_DIR}")


## 3. Countdown reward (vendored logic from `countdown_rewards.py`)

The RL signal is **`composite_reward`**: partial credit for a parseable `<answer>`, using exactly the puzzle numbers, and evaluating to the target. This is the same shaping as the cookbook—copied here so you can inspect and unit-test it without opening another file.


In [ ]:
import json
import re

def extract_answer(text: str) -> str | None:
    matches = list(re.finditer(r"<answer>(.*?)</answer>", text, flags=re.IGNORECASE | re.DOTALL))
    return matches[-1].group(1).strip() if matches else None

def safe_eval_equation(equation: str) -> float | None:
    cleaned = equation.replace(" ", "")
    if not re.match(r"^[\d+\-*/().]+$", cleaned):
        return None
    try:
        return float(eval(cleaned, {"__builtins__": None}, {}))
    except Exception:
        return None

def check_numbers_used(equation: str, numbers: list[int]) -> bool:
    used = sorted(int(x) for x in re.findall(r"\d+", equation))
    return used == sorted(numbers)

def parse_ground_truth(ground_truth: str | dict) -> tuple[list[int], int]:
    gt = json.loads(ground_truth) if isinstance(ground_truth, str) else ground_truth
    numbers = gt.get("numbers") or gt.get("nums")
    return list(numbers), int(gt["target"])

def composite_reward(response: str, ground_truth: str | dict) -> float:
    numbers, target = parse_ground_truth(ground_truth)
    equation = extract_answer(response)
    if equation is None:
        return 0.0
    score = 0.1
    numbers_valid = check_numbers_used(equation, numbers)
    if numbers_valid:
        score += 0.2
    result = safe_eval_equation(equation)
    if numbers_valid and result is not None and abs(result - target) < 1e-6:
        score += 0.7
    return score

# Quick sanity check on format-only credit
_demo = composite_reward("<answer>1+2+3</answer>", '{"numbers":[1,2,3],"target":6}')
print(f"demo reward (wrong target, valid format): {_demo}")


## 4. Load JSONL and carve a fixed eval split

The cookbook loads rows, then **deterministically** removes `eval_prompt_groups` rows (shuffled with `eval_seed`) from training. The same held-out prompts are reused at step 0, every `eval_interval`, and at the end—track **`eval/raw_reward`** for the learning curve.


In [ ]:
import random

def load_rows(path: Path) -> list[dict]:
    return [json.loads(line) for line in path.open() if line.strip()]

def carve_out_eval_rows(rows, eval_prompt_groups: int, eval_seed: int):
    if eval_prompt_groups == 0:
        return list(rows), [], []
    order = list(range(len(rows)))
    random.Random(eval_seed).shuffle(order)
    eval_indices = order[:eval_prompt_groups]
    eval_index_set = set(eval_indices)
    train_rows = [row for i, row in enumerate(rows) if i not in eval_index_set]
    eval_rows = [rows[i] for i in eval_indices]
    return train_rows, eval_rows, eval_indices

all_rows = load_rows(DATASET)
train_rows, eval_rows, eval_row_indices = carve_out_eval_rows(all_rows, EVAL_PROMPT_GROUPS, EVAL_SEED)

order = list(range(len(train_rows)))
if SHUFFLE:
    random.Random(SEED).shuffle(order)

print(f"total={len(all_rows)} train={len(train_rows)} eval={len(eval_rows)}")
print("sample training user message:", train_rows[0]["messages"][-1]["content"][:120], "...")


## 5. Small helpers used in the training loop

These match private helpers in `countdown_rl.py`: GRPO-style group advantages, serverless URL normalization, and truncation detection.


In [ ]:
import math

def serverless_base_url(base_url: str) -> str:
    root = base_url.rstrip("/")
    if root.endswith("/training/v1/serverless"):
        return root
    if root.endswith("/training/v1"):
        return f"{root}/serverless"
    return f"{root}/training/v1/serverless"

def group_relative_advantages(rewards: list[float], eps: float = 1e-8) -> list[float]:
    if len(rewards) <= 1:
        return [0.0] * len(rewards)
    mean = sum(rewards) / len(rewards)
    variance = sum((r - mean) ** 2 for r in rewards) / (len(rewards) - 1)
    std = math.sqrt(variance)
    if std < 1e-6:
        std = 1.0
    return [(r - mean) / (std + eps) for r in rewards]

def validate_length(what: str, length: int, max_seq_len: int) -> None:
    if length > max_seq_len:
        raise ValueError(f"{what} length {length} exceeds max_seq_len {max_seq_len}")

def is_truncated(seq, max_tokens: int | None = None) -> bool:
    reason = str(getattr(getattr(seq, "finish_reason", ""), "value", getattr(seq, "finish_reason", ""))).lower()
    if reason == "length" or "max_token" in reason:
        return True
    tokens = getattr(seq, "tokens", None) or []
    return max_tokens is not None and len(tokens) >= max_tokens

print("serverless URL:", serverless_base_url(API_BASE_URL))
print("GRPO advantages:", group_relative_advantages([0.1, 0.5, 0.9]))


## 6. Tokenizer, renderer, and serverless clients

**Tokenizer + renderer:** prompts are built client-side with the cookbook renderer (`build_generation_prompt` / `parse_response`).

**Service client:** one connection creates the LoRA trainer. No separate deployment—the pooled serverless trainer handles both forward/backward and (via snapshots) sampling.


In [ ]:
import tinker
from fireworks.training.sdk import FiretitanServiceClient, FiretitanSamplingParams
from training.renderer import get_renderer, get_text_content
import training.renderer  # registers kimi_k3 renderers
from training.utils.supervised import resolve_renderer_name
from training.utils.tokenizers import load_tokenizer
from training.utils import resolve_router_replay_enabled

tokenizer = load_tokenizer(TOKENIZER_MODEL)
renderer_name = resolve_renderer_name(TOKENIZER_MODEL, RENDERER_NAME)
renderer = get_renderer(renderer_name, tokenizer)

print(f"renderer={renderer_name}")


## 7. Paid-run gate + explicit loop

Everything below connects to Fireworks and performs sampling/training. Leave **`RUN_TRAINING = False`** until you intentionally spend credits.

When enabled, the notebook runs the same control flow as `ServerlessCountdownRL.run()`:

| Phase | What happens |
| --- | --- |
| Connect | `FiretitanServiceClient` + `create_lora_training_client` |
| Router Replay | resolve MoE vs dense via model metadata |
| Eval (optional) | save snapshot → sample held-out prompts → log `eval/raw_reward` |
| Each step | snapshot → sample `GROUP_SIZE` per prompt → score → drop zero-variance groups → build `tinker.Datum` → `forward_backward(..., importance_sampling)` → `optim_step` |
| Finish | save final sampler weights, plot rollout rewards |

Read the function bodies in the next cells—they are the cookbook `_evaluate` and `_step` logic, inlined with comments.


In [ ]:
RUN_TRAINING = False

if not RUN_TRAINING:
    print("Training disabled. Set RUN_TRAINING = True to connect and run the loop.")


In [ ]:
import time
from training.utils.rl.router_replay import build_r3_routing_matrices, validate_r3_routing_matrices

def next_batch(rows, order, row_cursor, batch_size):
    n = len(order)
    idx = [order[(row_cursor + i) % n] for i in range(batch_size)]
    return [rows[i] for i in idx], row_cursor + batch_size

def mean_loss(fb_output):
    metrics = getattr(fb_output, "metrics", None) or {}
    loss_sum = metrics.get("loss:sum")
    tokens = metrics.get("response_tokens") or metrics.get("num_loss_tokens") or 1.0
    return float(loss_sum) / max(float(tokens), 1.0) if loss_sum is not None else None

def evaluate(training_client, service, completed_steps, *, router_replay_enabled):
    started = time.time()
    save_name = f"cd-eval-{completed_steps:04d}"
    snapshot = training_client.save_weights_for_sampler(save_name).result().path
    prompts = [renderer.build_generation_prompt(row["messages"]) for row in eval_rows]
    for prompt in prompts:
        validate_length("eval prompt + max_sample_tokens", prompt.length + MAX_SAMPLE_TOKENS, MAX_SEQ_LEN)

    sampler = service.create_sampling_client(model_path=snapshot, tokenizer=tokenizer)
    try:
        params = FiretitanSamplingParams(
            max_tokens=MAX_SAMPLE_TOKENS,
            temperature=TEMPERATURE,
            stop=renderer.get_stop_sequences(),
        )
        results = []
        chunk = max(1, PROMPT_CONCURRENCY)
        for start in range(0, len(prompts), chunk):
            futures = [
                sampler.sample(prompt=p, num_samples=EVAL_GROUP_SIZE, sampling_params=params)
                for p in prompts[start : start + chunk]
            ]
            results.extend(f.result(timeout=SAMPLING_TIMEOUT_S) for f in futures)
    finally:
        sampler.close()

    rewards = []
    for row, result in zip(eval_rows, results, strict=True):
        for seq in getattr(result, "sequences", []) or []:
            tokens = list(getattr(seq, "tokens", []) or [])
            content = get_text_content(renderer.parse_response(tokens)[0])
            rewards.append(float(composite_reward(content, row["ground_truth"])))

    rec = {
        "completed_steps": completed_steps,
        "eval/raw_reward": sum(rewards) / len(rewards) if rewards else 0.0,
        "eval/samples": len(rewards),
        "perf/eval_wall_time": time.time() - started,
        "train/router_replay": router_replay_enabled,
    }
    print(f"eval step={completed_steps} raw_reward={rec['eval/raw_reward']:.3f} samples={len(rewards)}")
    return rec

def training_step(training_client, service, step, batch_rows, *, router_replay_enabled):
    t0 = time.time()
    save_name = f"{CHECKPOINT_NAME}-{step:04d}"
    snapshot = training_client.save_weights_for_sampler(save_name).result().path

    prompts = [renderer.build_generation_prompt(row["messages"]) for row in batch_rows]
    for prompt in prompts:
        validate_length("prompt + max_sample_tokens", prompt.length + MAX_SAMPLE_TOKENS, MAX_SEQ_LEN)

    sampler = service.create_sampling_client(model_path=snapshot, tokenizer=tokenizer)
    try:
        params = FiretitanSamplingParams(
            max_tokens=MAX_SAMPLE_TOKENS,
            temperature=TEMPERATURE,
            stop=renderer.get_stop_sequences(),
            include_routing_matrix=router_replay_enabled,
        )
        results = []
        chunk = max(1, PROMPT_CONCURRENCY)
        for start in range(0, len(prompts), chunk):
            futures = [
                sampler.sample(prompt=p, num_samples=GROUP_SIZE, sampling_params=params)
                for p in prompts[start : start + chunk]
            ]
            results.extend(f.result(timeout=SAMPLING_TIMEOUT_S) for f in futures)
    finally:
        sampler.close()

    datums = []
    raw_rewards = []
    filtered_rewards = []

    for result, prompt, row in zip(results, prompts, batch_rows, strict=True):
        tokens_g, logprobs_g, routing_g, rewards_g = [], [], [], []
        for seq in getattr(result, "sequences", []) or []:
            tokens = list(getattr(seq, "tokens", []) or [])
            logprobs = getattr(seq, "logprobs", None)
            if not tokens or logprobs is None or len(logprobs) != len(tokens):
                continue
            content = get_text_content(renderer.parse_response(tokens)[0])
            reward = float(composite_reward(content, row["ground_truth"]))
            tokens_g.append(tokens)
            logprobs_g.append([float(x) for x in logprobs])
            routing_matrices = getattr(seq, "routing_matrices", None)
            routing_g.append(list(routing_matrices) if routing_matrices is not None else None)
            rewards_g.append(reward)
            raw_rewards.append(reward)

        if len(set(rewards_g)) <= 1:
            continue  # GRPO: drop groups with no reward spread
        filtered_rewards.extend(rewards_g)

        advantages = group_relative_advantages(rewards_g)
        response_start = prompt.length - 1
        for tokens, logprobs, routing_matrices, advantage in zip(tokens_g, logprobs_g, routing_g, advantages, strict=True):
            model_input = prompt.append(tinker.EncodedTextChunk(tokens=tokens[:-1]))
            validate_length("training datum", model_input.length, MAX_SEQ_LEN)
            if router_replay_enabled:
                validate_r3_routing_matrices(
                    routing_matrices,
                    prompt_len=prompt.length,
                    model_input_len=model_input.length,
                )
                aligned = build_r3_routing_matrices(
                    routing_matrices,
                    prompt_len=prompt.length,
                    model_input_len=model_input.length,
                    completion_only=True,
                )
                model_input = model_input.model_copy(update={"routing_matrices": aligned})
            response_len = model_input.length - response_start
            datums.append(
                tinker.Datum(
                    model_input=model_input,
                    loss_fn_inputs={
                        "target_tokens": [0] * response_start + tokens,
                        "logprobs": [0.0] * response_start + logprobs,
                        "advantages": [0.0] * response_start + [advantage] * response_len,
                    },
                )
            )

    loss = None
    if datums:
        fb = training_client.forward_backward(datums, "importance_sampling").result()
        loss = mean_loss(fb)
        adam = tinker.AdamParams(
            learning_rate=LEARNING_RATE,
            beta1=0.9,
            beta2=0.95,
            eps=1e-12,
            weight_decay=0.0,
        )
        training_client.optim_step(adam).result()

    raw_mean = sum(raw_rewards) / len(raw_rewards) if raw_rewards else 0.0
    filt_mean = sum(filtered_rewards) / len(filtered_rewards) if filtered_rewards else 0.0
    filter_ratio = 1.0 - len(filtered_rewards) / len(raw_rewards) if raw_rewards else 0.0

    rec = {
        "step": step,
        "rollout/raw_reward": raw_mean,
        "rollout/filtered_reward": filt_mean,
        "rollout/filter_ratio": filter_ratio,
        "train/loss": loss,
        "train/trained": bool(datums),
        "train/router_replay": router_replay_enabled,
        "perf/step_wall_time": time.time() - t0,
    }
    print(
        f"step {step:02d} reward={raw_mean:.3f}/{filt_mean:.3f} "
        f"filter={filter_ratio:.1%} loss={'n/a' if loss is None else f'{loss:.4f}'}"
    )
    return rec

print("Defined evaluate() and training_step() — cookbook _evaluate / _step equivalents.")


In [ ]:
if RUN_TRAINING:
    if not api_key:
        raise RuntimeError("Set FIREWORKS_API_KEY before training.")

    RUN_DIR.mkdir(parents=True, exist_ok=True)
    metrics_path = RUN_DIR / "metrics.jsonl"
    eval_metrics_path = RUN_DIR / "eval_metrics.jsonl"

    service = FiretitanServiceClient(api_key=api_key, base_url=serverless_base_url(API_BASE_URL))
    training_client = service.create_lora_training_client(
        base_model=BASE_MODEL,
        rank=LORA_RANK,
        alpha=LORA_ALPHA,
    )

    router_replay_enabled = resolve_router_replay_enabled(
        requested=ROUTER_REPLAY,
        api_key=api_key,
        base_url=API_BASE_URL.rstrip("/"),
        additional_headers=None,
        base_model=BASE_MODEL,
    )
    if ROUTER_REPLAY and not router_replay_enabled:
        print(f"Router Replay skipped for dense model {BASE_MODEL}")

    row_cursor = 0
    records = []
    last_eval_step = None

    if EVAL_AT_START:
        rec = evaluate(training_client, service, 0, router_replay_enabled=router_replay_enabled)
        with eval_metrics_path.open("a") as handle:
            handle.write(json.dumps(rec) + "\n")
        last_eval_step = 0

    for step in range(STEPS):
        batch, row_cursor = next_batch(train_rows, order, row_cursor, PROMPT_GROUPS_PER_STEP)
        rec = training_step(training_client, service, step, batch, router_replay_enabled=router_replay_enabled)
        with metrics_path.open("a") as handle:
            handle.write(json.dumps(rec) + "\n")
        records.append(rec)

        completed = step + 1
        if EVAL_INTERVAL and completed % EVAL_INTERVAL == 0:
            erec = evaluate(training_client, service, completed, router_replay_enabled=router_replay_enabled)
            with eval_metrics_path.open("a") as handle:
                handle.write(json.dumps(erec) + "\n")
            last_eval_step = completed

    final_step = STEPS
    if EVAL_AT_END and last_eval_step != final_step:
        erec = evaluate(training_client, service, final_step, router_replay_enabled=router_replay_enabled)
        with eval_metrics_path.open("a") as handle:
            handle.write(json.dumps(erec) + "\n")

    final = training_client.save_weights_for_sampler(FINAL_CHECKPOINT_NAME).result()
    (RUN_DIR / "final_checkpoint.txt").write_text(f"{getattr(final, 'path', None)}\n")

    service.close()
    print(f"Finished {len(records)} training steps. Metrics: {metrics_path}")


## 8. Plot rollout rewards (after a run)

This mirrors the cookbook’s closing `reward_curve.png`: raw vs filtered rollout reward per optimizer step. **`eval/raw_reward`** in `eval_metrics.jsonl` is the metric to trust for generalization.


In [ ]:
from IPython.display import Image, display

if RUN_DIR.exists() and (RUN_DIR / "metrics.jsonl").exists():
    import matplotlib.pyplot as plt

    records = [json.loads(line) for line in (RUN_DIR / "metrics.jsonl").open() if line.strip()]
    steps = [r["step"] for r in records]
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(steps, [r["rollout/raw_reward"] for r in records], marker="o", label="raw_reward")
    ax.plot(steps, [r["rollout/filtered_reward"] for r in records], marker="s", linestyle="--", label="filtered_reward")
    ax.set_xlabel("optimizer step")
    ax.set_ylabel("composite_reward")
    ax.set_ylim(bottom=0.0)
    ax.legend()
    ax.grid(True, alpha=0.3)
    plot_path = RUN_DIR / "reward_curve.png"
    fig.savefig(plot_path, dpi=120)
    display(Image(filename=str(plot_path)))
else:
    print("No metrics yet — enable RUN_TRAINING and complete a run first.")


## What to read in the metrics

- **`rollout/raw_reward`**: average score on the training batch before GRPO filtering.
- **`rollout/filtered_reward`**: average over completions that actually contributed a non-zero group-relative advantage.
- **`rollout/filter_ratio`**: fraction of samples dropped because every completion in a prompt group scored the same.
- **`train/router_replay`**: whether MoE routing matrices were requested and replayed (dense models report false even when `ROUTER_REPLAY=True`).
- **`eval/raw_reward`**: fixed held-out prompts — use this as your learning curve.

For checkpointing, resume, promotion, and W&B, see the upstream [`countdown_rl.py` README](https://github.com/fw-ai/cookbook/tree/main/training/examples/serverless_rl); this notebook focuses on the core serverless RL loop spelled out step by step.
